In [56]:
# =============================================================================
# IMPORTS & SETTINGS
# =============================================================================
import os
import random
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from functools import partial

import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import scipy.optimize as opt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score, mean_squared_error, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import KNNImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesRegressor

warnings.filterwarnings('ignore')

RND_SEED = 42
random.seed(RND_SEED)
np.random.seed(RND_SEED)
torch.manual_seed(RND_SEED)

# Check environment: use Kaggle path if available, else local path
DATA_DIR = 'data/'
if os.path.exists('/kaggle/input/'):
    for root, dirs, files in os.walk('/kaggle/input/'):
        if 'train.csv' in files:
            DATA_DIR = root
            break


In [57]:
# =============================================================================
# CONSTANTS & THRESHOLD OPTIMIZER
# =============================================================================
TARGET_sii = 'sii'
TARGET_PC = 'PCIAT-PCIAT_Total'
TARGET_cols = [TARGET_sii, TARGET_PC]

class OptimizedRounder:
    def __init__(self):
        self.coef_ = 0
    
    def _kappa_loss(self, coef, X, y):
        X_p = np.copy(X)
        for i, pred in enumerate(X_p):
            if pred < coef[0]:
                X_p[i] = 0
            elif pred < coef[1]:
                X_p[i] = 1
            elif pred < coef[2]:
                X_p[i] = 2
            else:
                X_p[i] = 3
        ll = cohen_kappa_score(y, X_p, weights='quadratic')
        return -ll
        
    def fit(self, X, y):
        loss_partial = partial(self._kappa_loss, X=X, y=y)
        initial_coef = [31, 50, 80] # Start with medical boundaries
        self.coef_ = opt.minimize(loss_partial, initial_coef, method='nelder-mead')
        
    def predict(self, X, coef):
        X_p = np.copy(X)
        for i, pred in enumerate(X_p):
            if pred < coef[0]:
                X_p[i] = 0
            elif pred < coef[1]:
                X_p[i] = 1
            elif pred < coef[2]:
                X_p[i] = 2
            else:
                X_p[i] = 3
        return X_p.astype(int)


In [58]:
# =============================================================================
# DATA READING
# =============================================================================
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sample = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

all_cols = train.columns.to_list()
feature_cols = [col for col in all_cols if ('PCIAT' not in col) and (col not in ['sii', 'id', 'binned'])]
num_cols = train[feature_cols].select_dtypes(['number']).columns.to_list()
cat_cols = [col for col in feature_cols if 'Season' in col]

train[cat_cols] = train[cat_cols].replace({'Spring':1,'Summer':2,'Fall':3,'Winter':4}).fillna(0).astype('int')
test[cat_cols] = test[cat_cols].replace({'Spring':1,'Summer':2,'Fall':3,'Winter':4}).fillna(0).astype('int')


In [59]:
# =============================================================================
# FEATURE ENGINEERING: Deep Embeddings + Tabular Statistics
# =============================================================================
SEQ_LEN = 1000

class TSAE(nn.Module):
    def __init__(self, input_dim=4, embed_dim=16):
        super(TSAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(input_dim, 16, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),
            nn.Conv1d(16, embed_dim, kernel_size=5, stride=2, padding=2),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(embed_dim, 16, kernel_size=5, stride=2, padding=2, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose1d(16, input_dim, kernel_size=5, stride=2, padding=2, output_padding=1)
        )
        
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

def train_and_extract_ts_embeddings(dirname, is_train=True, trained_model=None):
    ids = os.listdir(dirname)
    
    all_data = []
    all_stats = []
    all_ids = []
    
    def read_file(filename):
        try:
            df = pd.read_parquet(os.path.join(dirname, filename, 'part-0.parquet'))
            df.dropna(inplace=True)
            if len(df) == 0: return None
            
            cols = ['X', 'Y', 'Z', 'enmo']
            data = df[cols].values.T # shape (4, N)
            
            # --- TABULAR STATISTICS ---
            means = np.mean(data, axis=1)
            stds = np.std(data, axis=1)
            maxs = np.max(data, axis=1)
            mins = np.min(data, axis=1)
            q25 = np.percentile(data, 25, axis=1)
            q75 = np.percentile(data, 75, axis=1)
            stats = np.concatenate([means, stds, maxs, mins, q25, q75]) # 24 stats
            
            # --- PREP FOR TSAE ---
            # Normalize
            data = (data - np.mean(data, axis=1, keepdims=True)) / (np.std(data, axis=1, keepdims=True) + 1e-8)
            if data.shape[1] < SEQ_LEN:
                pad_width = SEQ_LEN - data.shape[1]
                data = np.pad(data, ((0,0), (0, pad_width)), mode='constant')
            else:
                data = data[:, :SEQ_LEN]
                
            return filename.split('=')[1], data, stats
        except:
            return None

    with ThreadPoolExecutor() as executor:
        results = list(tqdm(executor.map(read_file, ids), total=len(ids), desc=f"Reading {dirname}"))
        
    for res in results:
        if res is not None:
            all_ids.append(res[0])
            all_data.append(res[1])
            all_stats.append(res[2])
            
    if len(all_data) == 0:
        return pd.DataFrame(columns=['id'] + [f'ts_embed_{i}' for i in range(16)] + [f'ts_stat_{i}' for i in range(24)]), trained_model
        
    tensor_data = torch.tensor(np.array(all_data), dtype=torch.float32)
    stats_data = np.array(all_stats)
    
    if is_train and trained_model is None:
        model = TSAE(input_dim=4, embed_dim=16)
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.MSELoss()
        dataset = TensorDataset(tensor_data, tensor_data)
        loader = DataLoader(dataset, batch_size=128, shuffle=True)
        
        print("Training TSAE Autoencoder...")
        model.train()
        for epoch in range(10):
            epoch_loss = 0
            for bx, by in loader:
                optimizer.zero_grad()
                _, decoded = model(bx)
                loss = criterion(decoded, by)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            print(f"TSAE Epoch {epoch+1}, Loss: {epoch_loss/len(loader):.4f}")
        trained_model = model
    else:
        model = trained_model
        
    model.eval()
    with torch.no_grad():
        encoded_all = []
        loader = DataLoader(TensorDataset(tensor_data), batch_size=128, shuffle=False)
        for bx in loader:
            encoded, _ = model(bx[0])
            emb = encoded.mean(dim=-1).numpy()
            encoded_all.append(emb)
        embeddings = np.vstack(encoded_all)
        
    # Combine Embeddings + Tabular Stats
    combined_features = np.hstack([embeddings, stats_data])
    
    emb_cols = [f'ts_embed_{i}' for i in range(16)]
    stat_cols = [f'ts_stat_{i}' for i in range(24)]
    
    df_emb = pd.DataFrame(combined_features, columns=emb_cols + stat_cols)
    df_emb.insert(0, 'id', all_ids)
    
    return df_emb, trained_model

train_ts_df, trained_tsae = train_and_extract_ts_embeddings(os.path.join(DATA_DIR, "series_train.parquet"), is_train=True)
test_ts_df, _ = train_and_extract_ts_embeddings(os.path.join(DATA_DIR, "series_test.parquet"), is_train=False, trained_model=trained_tsae)

train = pd.merge(train, train_ts_df, how="left", on='id')
test = pd.merge(test, test_ts_df, how="left", on='id')

ts_cols = [f'ts_embed_{i}' for i in range(16)] + [f'ts_stat_{i}' for i in range(24)]
num_cols += ts_cols
feature_cols += ts_cols

train[ts_cols] = train[ts_cols].fillna(0)
test[ts_cols] = test[ts_cols].fillna(0)


Reading data/series_train.parquet: 100%|██████████| 996/996 [00:08<00:00, 120.69it/s]


Training TSAE Autoencoder...
TSAE Epoch 1, Loss: 2.8418
TSAE Epoch 2, Loss: 2.7473
TSAE Epoch 3, Loss: 2.5806
TSAE Epoch 4, Loss: 2.2257
TSAE Epoch 5, Loss: 1.8841
TSAE Epoch 6, Loss: 1.5839
TSAE Epoch 7, Loss: 1.3511
TSAE Epoch 8, Loss: 1.1388
TSAE Epoch 9, Loss: 0.9588
TSAE Epoch 10, Loss: 0.8122


Reading data/series_test.parquet: 100%|██████████| 2/2 [00:00<00:00, 46.78it/s]


In [60]:
# =============================================================================
# IMPUTATION AND SCALING
# =============================================================================
imputer = KNNImputer(n_neighbors=10)
train[num_cols] = imputer.fit_transform(train[num_cols])
test[num_cols] = imputer.transform(test[num_cols])

scaler = StandardScaler()
train[num_cols] = scaler.fit_transform(train[num_cols])
test[num_cols] = scaler.transform(test[num_cols])

train_missing = train[train[TARGET_sii].isna()].copy().reset_index(drop=True)
train_labeled = train.dropna(subset=[TARGET_sii]).copy().reset_index(drop=True)

def get_sample_weights(y):
    y_series = pd.Series(y)
    bins = pd.cut(y_series, bins=10, labels=False)
    bin_counts = bins.value_counts(normalize=True)
    weight_map = (1 / bin_counts).to_dict()
    weights = bins.map(weight_map)
    return weights / weights.mean()

sample_weights = get_sample_weights(train_labeled[TARGET_PC]).values


In [61]:
# =============================================================================
# MODEL DEFINITIONS (LGBM, XGBoost, CatBoost, PyTorch MLP)
# =============================================================================

class SimpleMLP(nn.Module):
    def __init__(self, input_dim):
        super(SimpleMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_mlp(X_tr, y_tr, X_va, y_va, w_tr, epochs=150, lr=0.01):
    model = SimpleMLP(X_tr.shape[1])
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=10, factor=0.5)
    criterion = nn.MSELoss(reduction='none')
    
    X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
    y_tr_t = torch.tensor(y_tr, dtype=torch.float32)
    w_tr_t = torch.tensor(w_tr, dtype=torch.float32)
    X_va_t = torch.tensor(X_va, dtype=torch.float32)
    y_va_t = torch.tensor(y_va, dtype=torch.float32)
    
    dataset = TensorDataset(X_tr_t, y_tr_t, w_tr_t)
    loader = DataLoader(dataset, batch_size=256, shuffle=True, drop_last=True)
    
    best_loss = float('inf')
    patience_counter = 0
    best_weights = None
    
    for epoch in range(epochs):
        model.train()
        for bx, by, bw in loader:
            optimizer.zero_grad()
            preds = model(bx)
            loss = (criterion(preds, by) * bw).mean()
            loss.backward()
            optimizer.step()
            
        model.eval()
        with torch.no_grad():
            va_preds = model(X_va_t)
            val_loss = nn.MSELoss()(va_preds, y_va_t).item()
            
        scheduler.step(val_loss)
        
        if val_loss < best_loss:
            best_loss = val_loss
            patience_counter = 0
            best_weights = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= 20:
                break
                
    if best_weights is not None:
        model.load_state_dict(best_weights)
        
    model.eval()
    with torch.no_grad():
        final_va_preds = model(X_va_t).numpy()
        
    return model, final_va_preds

def predict_mlp(model, X):
    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(X, dtype=torch.float32)).numpy()
    return preds

# Ensemble Weight Optimizer
def find_best_weights(preds_list, y_true):
    def loss_func(weights):
        weights = np.array(weights) / np.sum(weights)
        blended = np.zeros_like(y_true, dtype=float)
        for w, p in zip(weights, preds_list):
            blended += w * p
        return mean_squared_error(y_true, blended)
        
    init_weights = [1.0/len(preds_list)] * len(preds_list)
    bounds = [(0, 1)] * len(preds_list)
    res = opt.minimize(loss_func, init_weights, bounds=bounds, method='SLSQP')
    return res.x / np.sum(res.x)


In [62]:
# =============================================================================
# CROSS VALIDATION & IN-FOLD PSEUDO-LABELING
# =============================================================================
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RND_SEED)

oof_preds_lgb = np.zeros(len(train_labeled))
oof_preds_xgb = np.zeros(len(train_labeled))
oof_preds_cb = np.zeros(len(train_labeled))
oof_preds_mlp = np.zeros(len(train_labeled))
oof_preds_ridge = np.zeros(len(train_labeled))
oof_preds_et = np.zeros(len(train_labeled))

test_preds_lgb = np.zeros(len(test))
test_preds_xgb = np.zeros(len(test))
test_preds_cb = np.zeros(len(test))
test_preds_mlp = np.zeros(len(test))
test_preds_ridge = np.zeros(len(test))
test_preds_et = np.zeros(len(test))

X = train_labeled[feature_cols].values
y = train_labeled[TARGET_PC].values
y_sii = train_labeled[TARGET_sii].values
W = sample_weights

X_unlabeled = train_missing[feature_cols].values
X_test = test[feature_cols].values

for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y_sii)):
    print(f"--- Fold {fold+1} ---")
    X_tr, y_tr, w_tr = X[trn_idx], y[trn_idx], W[trn_idx]
    X_va, y_va = X[val_idx], y[val_idx]
    
    # --- PHASE 1: Train initial models on labeled data ---
    # Enhanced Regularization
    model_lgb = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.03, max_depth=6, subsample=0.7, colsample_bytree=0.7, reg_alpha=0.5, reg_lambda=0.5, objective='tweedie', tweedie_variance_power=1.5, random_state=RND_SEED, verbose=-1)
    model_lgb.fit(X_tr, y_tr, sample_weight=w_tr)
    
    model_xgb = xgb.XGBRegressor(n_estimators=400, learning_rate=0.03, max_depth=6, subsample=0.7, colsample_bytree=0.7, reg_alpha=0.5, reg_lambda=0.5, objective='reg:tweedie', tweedie_variance_power=1.5, random_state=RND_SEED)
    model_xgb.fit(X_tr, y_tr, sample_weight=w_tr)
    
    model_cb = cb.CatBoostRegressor(iterations=400, learning_rate=0.03, depth=6, loss_function='Tweedie:variance_power=1.5', l2_leaf_reg=3, random_seed=RND_SEED, verbose=0)
    model_cb.fit(X_tr, y_tr, sample_weight=w_tr)
    
    model_mlp, _ = train_mlp(X_tr, y_tr, X_va, y_va, w_tr)
    
    model_ridge = Ridge(alpha=1.0, random_state=RND_SEED)
    model_ridge.fit(X_tr, y_tr, sample_weight=w_tr)
    
    model_et = ExtraTreesRegressor(n_estimators=300, max_depth=10, min_samples_leaf=4, n_jobs=-1, random_state=RND_SEED)
    model_et.fit(X_tr, y_tr, sample_weight=w_tr)
    
    # --- PHASE 2: Predict on Unlabeled data & Filter Confident Samples ---
    unlabeled_pred_lgb = np.clip(model_lgb.predict(X_unlabeled), 0, 100)
    unlabeled_pred_xgb = np.clip(model_xgb.predict(X_unlabeled), 0, 100)
    unlabeled_pred_cb = np.clip(model_cb.predict(X_unlabeled), 0, 100)
    unlabeled_pred_mlp = np.clip(predict_mlp(model_mlp, X_unlabeled), 0, 100)
    unlabeled_pred_ridge = np.clip(model_ridge.predict(X_unlabeled), 0, 100)
    unlabeled_pred_et = np.clip(model_et.predict(X_unlabeled), 0, 100)
    
    unlabeled_preds = np.vstack([unlabeled_pred_lgb, unlabeled_pred_xgb, unlabeled_pred_cb, unlabeled_pred_mlp, unlabeled_pred_ridge, unlabeled_pred_et])
    unlabeled_var = np.var(unlabeled_preds, axis=0)
    unlabeled_mean = np.mean(unlabeled_preds, axis=0)
    
    confidence_threshold = np.percentile(unlabeled_var, 30)
    confident_mask = unlabeled_var <= confidence_threshold
    
    X_confident = X_unlabeled[confident_mask]
    y_confident = unlabeled_mean[confident_mask]
    w_confident = get_sample_weights(y_confident)
    
    print(f"Fold {fold+1}: Adding {len(X_confident)} confident pseudo-labels out of {len(X_unlabeled)}")
    
    # --- PHASE 3: Retrain models on augmented data ---
    X_tr_aug = np.vstack([X_tr, X_confident])
    y_tr_aug = np.concatenate([y_tr, y_confident])
    w_tr_aug = np.concatenate([w_tr, w_confident])
    
    model_lgb.fit(X_tr_aug, y_tr_aug, sample_weight=w_tr_aug)
    model_xgb.fit(X_tr_aug, y_tr_aug, sample_weight=w_tr_aug)
    model_cb.fit(X_tr_aug, y_tr_aug, sample_weight=w_tr_aug)
    model_mlp, _ = train_mlp(X_tr_aug, y_tr_aug, X_va, y_va, w_tr_aug)
    model_ridge.fit(X_tr_aug, y_tr_aug, sample_weight=w_tr_aug)
    model_et.fit(X_tr_aug, y_tr_aug, sample_weight=w_tr_aug)
    
    # --- PHASE 4: Evaluate on Validation fold (OOF) ---
    oof_preds_lgb[val_idx] = np.clip(model_lgb.predict(X_va), 0, 100)
    oof_preds_xgb[val_idx] = np.clip(model_xgb.predict(X_va), 0, 100)
    oof_preds_cb[val_idx] = np.clip(model_cb.predict(X_va), 0, 100)
    oof_preds_mlp[val_idx] = np.clip(predict_mlp(model_mlp, X_va), 0, 100)
    oof_preds_ridge[val_idx] = np.clip(model_ridge.predict(X_va), 0, 100)
    oof_preds_et[val_idx] = np.clip(model_et.predict(X_va), 0, 100)

    # --- PHASE 5: Predict on Test fold ---
    test_preds_lgb += np.clip(model_lgb.predict(X_test), 0, 100) / n_splits
    test_preds_xgb += np.clip(model_xgb.predict(X_test), 0, 100) / n_splits
    test_preds_cb += np.clip(model_cb.predict(X_test), 0, 100) / n_splits
    test_preds_mlp += np.clip(predict_mlp(model_mlp, X_test), 0, 100) / n_splits
    test_preds_ridge += np.clip(model_ridge.predict(X_test), 0, 100) / n_splits
    test_preds_et += np.clip(model_et.predict(X_test), 0, 100) / n_splits

# --- ENSEMBLE WEIGHT OPTIMIZATION ---
preds_list = [oof_preds_lgb, oof_preds_xgb, oof_preds_cb, oof_preds_mlp, oof_preds_ridge, oof_preds_et]
opt_weights = find_best_weights(preds_list, y)
print(f"\nOptimal Ensemble Weights: LGBM: {opt_weights[0]:.3f}, XGB: {opt_weights[1]:.3f}, CB: {opt_weights[2]:.3f}, MLP: {opt_weights[3]:.3f}, Ridge: {opt_weights[4]:.3f}, ET: {opt_weights[5]:.3f}")

oof_blended = np.zeros_like(y, dtype=float)
for w, p in zip(opt_weights, preds_list):
    oof_blended += w * p

test_blended = (
    opt_weights[0] * test_preds_lgb +
    opt_weights[1] * test_preds_xgb +
    opt_weights[2] * test_preds_cb +
    opt_weights[3] * test_preds_mlp +
    opt_weights[4] * test_preds_ridge +
    opt_weights[5] * test_preds_et
)


--- Fold 1 ---
Fold 1: Adding 370 confident pseudo-labels out of 1224
--- Fold 2 ---
Fold 2: Adding 367 confident pseudo-labels out of 1224
--- Fold 3 ---
Fold 3: Adding 368 confident pseudo-labels out of 1224
--- Fold 4 ---
Fold 4: Adding 367 confident pseudo-labels out of 1224
--- Fold 5 ---
Fold 5: Adding 367 confident pseudo-labels out of 1224

Optimal Ensemble Weights: LGBM: 0.116, XGB: 0.401, CB: 0.000, MLP: 0.037, Ridge: 0.000, ET: 0.446


In [63]:
# =============================================================================
# THRESHOLD OPTIMIZATION & FINAL EVALUATION
# =============================================================================
optR = OptimizedRounder()
optR.fit(oof_blended, y_sii)
best_thresholds = optR.coef_.x

print(f"\nOptimized Medical Thresholds: {best_thresholds}")

oof_avg_classes = optR.predict(oof_blended, best_thresholds)
qwk_score = cohen_kappa_score(y_sii, oof_avg_classes, weights='quadratic')

print("\n=== FINAL OOF EVALUATION SUMMARY ===")
print(f"OOF QWK Score: {qwk_score:.4f}")
print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_sii, oof_avg_classes, target_names=['None', 'Mild', 'Moderate', 'Severe']))
print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_sii, oof_avg_classes))

# --- GENERATE TEST SUBMISSION ---
test_classes = optR.predict(test_blended, best_thresholds)
submission = pd.DataFrame({
    'id': test['id'],
    'sii': test_classes
})
submission.to_csv('submission.csv', index=False)
print("\nSubmission file successfully saved to 'submission.csv'.")



Optimized Medical Thresholds: [31.88485686 42.51854313 84.65836483]

=== FINAL OOF EVALUATION SUMMARY ===
OOF QWK Score: 0.4614

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

        None       0.75      0.77      0.76      1594
        Mild       0.35      0.30      0.32       730
    Moderate       0.33      0.42      0.37       378
      Severe       0.00      0.00      0.00        34

    accuracy                           0.58      2736
   macro avg       0.36      0.37      0.36      2736
weighted avg       0.57      0.58      0.58      2736


=== CONFUSION MATRIX ===
[[1220  266  108    0]
 [ 322  217  191    0]
 [  90  128  160    0]
 [   0    8   26    0]]

Submission file successfully saved to 'submission.csv'.
